### Setup and Load Data from First Notebook

In [ ]:
from IPython.utils import io

with io.capture_output():
    %run ../01_eda_exploratory_data_analysis/01_data_overview.ipynb

print("✅ Connected to 01_data_overview.ipynb")

### Comprehensive Statistical Summary

In [ ]:
# ====================================================================
# 📊 COMPREHENSIVE STATISTICAL SUMMARY
# ====================================================================

print("=" * 60)
print("📊 COMPREHENSIVE STATISTICAL SUMMARY")
print("=" * 60)

# All numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n📋 Numeric Columns ({len(numeric_cols)}):")
for col in numeric_cols:
    print(f"  - {col}")

# Detailed statistics
display(df[numeric_cols].describe())

# Additional statistics for price
if price_col:
    data = df[price_col].dropna()
    print("\n📈 ADDITIONAL PRICE STATISTICS:")
    print(f"  Mean: {data.mean():.4f}")
    print(f"  Median: {data.median():.4f}")
    print(f"  Mode: {data.mode().iloc[0]:.4f}" if not data.mode().empty else "  Mode: N/A")
    print(f"  Std Dev: {data.std():.4f}")
    print(f"  Variance: {data.var():.4f}")
    print(f"  Range: {data.max() - data.min():.4f}")
    print(f"  IQR: {data.quantile(0.75) - data.quantile(0.25):.4f}")
    print(f"  CV (Std/Mean): {data.std()/data.mean():.4f}")
    
    # Percentiles
    print(f"\n📊 PRICE PERCENTILES:")
    for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
        print(f"  {p:2d}th: {data.quantile(p/100):.4f}")

### Distribution Analysis

In [ ]:
# ====================================================================
# 📊 DISTRIBUTION ANALYSIS
# ====================================================================

# Import required statistical functions
from scipy.stats import skew, kurtosis, jarque_bera, shapiro, normaltest

if price_col:
    data = df[price_col].dropna()
    returns = df['returns'].dropna() if 'returns' in df.columns else None
    
    print("=" * 60)
    print("📊 DISTRIBUTION ANALYSIS")
    print("=" * 60)
    
    # =================================================================
    # PRICE DISTRIBUTION
    # =================================================================
    print("\n📈 PRICE DISTRIBUTION:")
    print("-" * 40)
    
    skewness = skew(data)
    kurt = kurtosis(data)
    
    print(f"  Skewness: {skewness:.4f}")
    if abs(skewness) < 0.5:
        print("    → Approximately Symmetric (Balanced distribution)")
    elif skewness > 0:
        print("    → Positively Skewed (Right-tailed, more high values)")
    else:
        print("    → Negatively Skewed (Left-tailed, more low values)")
    
    print(f"  Kurtosis: {kurt:.4f}")
    if kurt > 3:
        print("    → Leptokurtic (Heavy tails, more outliers/extreme values)")
    elif kurt < 3:
        print("    → Platykurtic (Light tails, fewer outliers)")
    else:
        print("    → Mesokurtic (Normal-like distribution)")
    
    print(f"\n  Additional Price Stats:")
    print(f"    Mean: {data.mean():.4f}")
    print(f"    Median: {data.median():.4f}")
    print(f"    Std Dev: {data.std():.4f}")
    print(f"    Range: {data.max() - data.min():.4f}")
    
    # =================================================================
    # RETURNS DISTRIBUTION
    # =================================================================
    if returns is not None and len(returns) > 0:
        print("\n📈 RETURNS DISTRIBUTION:")
        print("-" * 40)
        
        skewness_r = skew(returns)
        kurt_r = kurtosis(returns)
        
        print(f"  Skewness: {skewness_r:.4f}")
        if abs(skewness_r) < 0.5:
            print("    → Approximately Symmetric")
        elif skewness_r > 0:
            print("    → Positively Skewed (More positive extreme returns)")
        else:
            print("    → Negatively Skewed (More negative extreme returns)")
        
        print(f"  Kurtosis: {kurt_r:.4f}")
        if kurt_r > 3:
            print("    → Leptokurtic (Fat tails, frequent extreme events)")
        elif kurt_r < 3:
            print("    → Platykurtic (Thin tails, fewer extreme events)")
        else:
            print("    → Mesokurtic (Normal-like)")
        
        print(f"\n  Additional Returns Stats (%):")
        print(f"    Mean: {returns.mean():.6f}")
        print(f"    Median: {returns.median():.6f}")
        print(f"    Std Dev: {returns.std():.6f}")
        print(f"    Min: {returns.min():.6f}")
        print(f"    Max: {returns.max():.6f}")
    
    # =================================================================
    # NORMALITY TESTS
    # =================================================================
    print("\n🔍 NORMALITY TESTS:")
    print("-" * 40)
    
    # 1. Jarque-Bera test
    try:
        jb_stat, jb_p = jarque_bera(data)
        print(f"  Jarque-Bera:")
        print(f"    Test Statistic: {jb_stat:.4f}")
        print(f"    P-value: {jb_p:.4f}")
        if jb_p > 0.05:
            print("    → Fail to reject normality (data appears normal)")
        else:
            print("    → Reject normality (data not normal)")
    except Exception as e:
        print(f"  Jarque-Bera: Error - {e}")
    
    # 2. D'Agostino's K^2 test
    try:
        dag_stat, dag_p = normaltest(data)
        print(f"\n  D'Agostino's K^2:")
        print(f"    Test Statistic: {dag_stat:.4f}")
        print(f"    P-value: {dag_p:.4f}")
        if dag_p > 0.05:
            print("    → Fail to reject normality (data appears normal)")
        else:
            print("    → Reject normality (data not normal)")
    except Exception as e:
        print(f"  D'Agostino's K^2: Error - {e}")
    
    # 3. Shapiro-Wilk test (limited to 5000 samples)
    try:
        if len(data) <= 5000:
            sample_size = min(5000, len(data))
            sample_data = data.sample(n=sample_size, random_state=42) if len(data) > sample_size else data
            shapiro_stat, shapiro_p = shapiro(sample_data)
            print(f"\n  Shapiro-Wilk:")
            print(f"    Test Statistic: {shapiro_stat:.4f}")
            print(f"    P-value: {shapiro_p:.4f}")
            print(f"    Sample Size: {sample_size}")
            if shapiro_p > 0.05:
                print("    → Fail to reject normality (data appears normal)")
            else:
                print("    → Reject normality (data not normal)")
    except Exception as e:
        print(f"  Shapiro-Wilk: Error - {e}")
    
    # =================================================================
    # SUMMARY INTERPRETATION
    # =================================================================
    print("\n📌 INTERPRETATION SUMMARY:")
    print("-" * 40)
    
    # Check if data is normal based on majority of tests
    normal_tests = []
    try:
        if jb_p > 0.05:
            normal_tests.append(True)
        else:
            normal_tests.append(False)
    except:
        pass
    
    try:
        if dag_p > 0.05:
            normal_tests.append(True)
        else:
            normal_tests.append(False)
    except:
        pass
    
    if normal_tests:
        normal_count = sum(normal_tests)
        total_tests = len(normal_tests)
        
        if normal_count / total_tests > 0.5:
            print("  ✅ Data appears to follow a normal distribution")
            print(f"     {normal_count}/{total_tests} tests indicate normality")
        else:
            print("  ⚠️  Data deviates from normal distribution")
            print(f"     {normal_count}/{total_tests} tests indicate normality")
    
    # Tail risk assessment
    if returns is not None:
        print(f"\n  📊 Tail Risk Assessment:")
        extreme_positive = (returns > 3 * returns.std()).sum()
        extreme_negative = (returns < -3 * returns.std()).sum()
        total_extreme = extreme_positive + extreme_negative
        pct_extreme = (total_extreme / len(returns)) * 100
        
        print(f"    Extreme Events (>3σ): {total_extreme} ({pct_extreme:.2f}%)")
        print(f"    Extreme Positive: {extreme_positive}")
        print(f"    Extreme Negative: {extreme_negative}")
        
        if pct_extreme > 1:
            print("    → Significant tail risk detected")
        else:
            print("    → Limited tail risk")
    
    print("\n" + "=" * 60)

Returns Analysis

In [ ]:
# ====================================================================
# 📊 RETURNS ANALYSIS
# ====================================================================

if price_col:
    # Calculate returns if not already done
    if 'returns' not in df.columns:
        df['returns'] = df[price_col].pct_change() * 100
        df['log_returns'] = np.log(df[price_col] / df[price_col].shift(1)) * 100
        df['cumulative'] = (df[price_col] / df[price_col].iloc[0] - 1) * 100
    
    returns = df['returns'].dropna()
    log_returns = df['log_returns'].dropna()
    cumulative = df['cumulative']
    
    print("=" * 60)
    print("📊 RETURNS ANALYSIS")
    print("=" * 60)
    
    print("\n📈 SIMPLE RETURNS (%):")
    print(f"  Mean: {returns.mean():.6f}")
    print(f"  Median: {returns.median():.6f}")
    print(f"  Std Dev: {returns.std():.6f}")
    print(f"  Min: {returns.min():.6f}")
    print(f"  Max: {returns.max():.6f}")
    print(f"  Skewness: {skew(returns):.4f}")
    print(f"  Kurtosis: {kurtosis(returns):.4f}")
    
    print("\n📈 LOG RETURNS (%):")
    print(f"  Mean: {log_returns.mean():.6f}")
    print(f"  Median: {log_returns.median():.6f}")
    print(f"  Std Dev: {log_returns.std():.6f}")
    print(f"  Min: {log_returns.min():.6f}")
    print(f"  Max: {log_returns.max():.6f}")
    
    # Performance metrics
    print("\n📊 PERFORMANCE METRICS:")
    total_return = cumulative.iloc[-1]
    avg_daily_return = returns.mean()
    volatility = returns.std()
    
    print(f"  Total Return: {total_return:.2f}%")
    print(f"  Annualized Return (252 days): {avg_daily_return * 252:.2f}%")
    print(f"  Annualized Volatility: {volatility * np.sqrt(252):.2f}%")
    
    # Sharpe Ratio (assuming 0% risk-free rate)
    sharpe = returns.mean() / returns.std() if returns.std() > 0 else 0
    sharpe_annual = sharpe * np.sqrt(252)
    print(f"  Sharpe Ratio (Daily): {sharpe:.4f}")
    print(f"  Sharpe Ratio (Annualized): {sharpe_annual:.4f}")
    
    # Win/Loss analysis
    positive = (returns > 0).sum()
    negative = (returns < 0).sum()
    zero = (returns == 0).sum()
    
    print(f"\n📊 WIN/LOSS ANALYSIS:")
    print(f"  Positive Days: {positive} ({positive/len(returns)*100:.1f}%)")
    print(f"  Negative Days: {negative} ({negative/len(returns)*100:.1f}%)")
    print(f"  Zero Days: {zero} ({zero/len(returns)*100:.1f}%)")
    
    # Average wins and losses
    avg_win = returns[returns > 0].mean() if (returns > 0).sum() > 0 else 0
    avg_loss = returns[returns < 0].mean() if (returns < 0).sum() > 0 else 0
    print(f"\n  Average Win: {avg_win:.4f}%")
    print(f"  Average Loss: {avg_loss:.4f}%")
    print(f"  Win/Loss Ratio: {abs(avg_win/avg_loss):.2f}" if avg_loss != 0 else "  Win/Loss Ratio: N/A")

Correlation Analysis

In [ ]:
# ====================================================================
# 📊 CORRELATION ANALYSIS
# ====================================================================

print("=" * 60)
print("📊 CORRELATION ANALYSIS")
print("=" * 60)

# Select key columns for correlation
corr_cols = []
for col in ['open', 'high', 'low', 'close', 'volume', 'returns', 'log_returns']:
    if col in df.columns:
        corr_cols.append(col)

# Add technical indicators
for col in ['RSI_14', 'MACD', 'ATR_14']:
    if col in df.columns:
        corr_cols.append(col)

if len(corr_cols) > 1:
    # Correlation matrix
    corr_matrix = df[corr_cols].corr()
    
    print("\n📈 CORRELATION MATRIX:")
    display(corr_matrix)
    
    # High correlations
    print("\n🔍 HIGH CORRELATIONS (>0.8 or <-0.8):")
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            val = corr_matrix.iloc[i, j]
            if abs(val) > 0.8:
                print(f"  {corr_matrix.columns[i]:15s} ↔ {corr_matrix.columns[j]:15s} : {val:.4f}")
    
    # Correlation heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=0.5, ax=ax)
    ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
Visualization 

In [ ]:
# ====================================================================
# 📊 STATISTICAL VISUALIZATIONS
# ====================================================================

if price_col:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. Price Distribution
    axes[0, 0].hist(df[price_col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='blue')
    axes[0, 0].axvline(x=df[price_col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df[price_col].mean():.2f}')
    axes[0, 0].axvline(x=df[price_col].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df[price_col].median():.2f}')
    axes[0, 0].set_title('Price Distribution', fontweight='bold')
    axes[0, 0].set_xlabel('Price ($)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Returns Distribution
    if 'returns' in df.columns:
        axes[0, 1].hist(df['returns'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='orange')
        axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
        axes[0, 1].axvline(x=df['returns'].mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {df["returns"].mean():.4f}%')
        axes[0, 1].set_title('Returns Distribution', fontweight='bold')
        axes[0, 1].set_xlabel('Returns (%)')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. QQ Plot
    if 'returns' in df.columns:
        stats.probplot(df['returns'].dropna(), dist="norm", plot=axes[0, 2])
        axes[0, 2].set_title('Q-Q Plot (Returns)', fontweight='bold')
        axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Box Plot
    if 'returns' in df.columns:
        axes[1, 0].boxplot([df['returns'].dropna(), df['log_returns'].dropna()], 
                           labels=['Returns', 'Log Returns'])
        axes[1, 0].set_title('Returns Box Plot', fontweight='bold')
        axes[1, 0].set_ylabel('Return (%)')
        axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Cumulative Returns
    if 'cumulative' in df.columns:
        axes[1, 1].plot(df.index, df['cumulative'], color='green', linewidth=1.5)
        axes[1, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
        axes[1, 1].set_title('Cumulative Returns', fontweight='bold')
        axes[1, 1].set_ylabel('Cumulative Return (%)')
        axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Rolling Volatility
    if 'returns' in df.columns and len(df) > 20:
        rolling_vol = df['returns'].rolling(20).std()
        axes[1, 2].plot(df.index, rolling_vol, color='purple', linewidth=1.5)
        axes[1, 2].set_title('20-Day Rolling Volatility', fontweight='bold')
        axes[1, 2].set_ylabel('Volatility (%)')
        axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()